# Exp 36: Model comparison

Card: [exp36_model_comparison.md](../../experiments/exp36_model_comparison.md)


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, Config, CVResult
from core.gradcam import GradCamFigures, compare
from core.predict import best_f1

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    N_SAMPLE = 2  # images per VIP comparison figure
else:
    N_SAMPLE = 16
SEED = 0
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
MODELS = {
    "exp26 ResNet-18": Config.from_toml(
        CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml"
    ),
    "EfficientNet-B0": Config.from_toml(
        CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.toml"
    ),
    "RegNetY-016": Config.from_toml(
        CONFIG_DIR / "exp33_regnety_016" / "regnety_016.toml"
    ),
    "ConvNeXt-Tiny": Config.from_toml(
        CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.toml"
    ),
}
TIMM_RESULTS = "https://raw.githubusercontent.com/huggingface/pytorch-image-models/main/results/results-imagenet.csv"
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp36_model_comparison"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; exp26 keeps orange from earlier cards
SERIES = dict(zip(MODELS, ["#eb6834", "#2a78d6", "#1baf7a", "#eda100"]))
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

## ImageNet scores


In [ ]:
# timm evaluation at 224 px, our training size
imagenet = pd.read_csv(TIMM_RESULTS)
weights = {c.timm_model: name for name, c in MODELS.items()}
imagenet = (
    imagenet[imagenet["model"].isin(weights) & (imagenet["img_size"] == 224)]
    .assign(name=lambda d: d["model"].map(weights))
    .set_index("name")
    .loc[list(MODELS), ["model", "top1", "top5"]]
)
imagenet

## Precision-recall curves


In [ ]:
results = {name: CVResult.load(c) for name, c in MODELS.items()}
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.8), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.suptitle("Out-of-fold rows of folds 0–4", color="#52514e")
fig.tight_layout()
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Thresholds

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
thresholds = pd.concat({name: best_f1(r.labels, r.probs)[["threshold", "f1"]] for name, r in results.items()}, axis=1)
thresholds.loc["Macro F1"] = [thresholds[col].mean() if col[1] == "f1" else float("nan") for col in thresholds.columns]
thresholds.round(3)

## Grad-CAM VIP: four models


In [ ]:
cams = {name: GradCamFigures(c) for name, c in MODELS.items()}
for cls in VIP:
    rows = cams["exp26 ResNet-18"].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = compare(cams, cls, rows)
plt.show()

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)